In [1]:
# Install required libraries
!pip install transformers datasets accelerate trl peft bitsandbytes evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.is_available())
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

True
GPU: Tesla T4


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "tiiuae/falcon-7b-instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Pad token set to: {tokenizer.eos_token}")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# Safely set pad_token_id
if getattr(model.config, "pad_token_id", None) is None:
    model.config.pad_token_id = tokenizer.pad_token_id

print("✅ Model and tokenizer loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

✅ Model and tokenizer loaded successfully


In [4]:
def generate_response(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test it
prompt = "Explain reinforcement learning simply"
print(generate_response(prompt))

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Explain reinforcement learning simply
Reinforcement learning is a branch of machine learning that involves learning by trial and error. In this process, an agent interacts with an environment and tries to find ways to maximize its reward or minimize its penalty by iteratively making decisions. The agent uses feedback to learn from its mistakes and refine its strategies over time.


In [5]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})


In [6]:
def format_prompt(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""
    return {"text": prompt}

dataset = dataset.map(format_prompt)
print("Formatting complete")

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

Formatting complete


In [7]:
from datasets import load_dataset

# Load the preference dataset
pref_dataset = load_dataset("Anthropic/hh-rlhf")
print(pref_dataset)

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 160800
    })
    test: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 8552
    })
})


In [8]:
# Use a small subset to avoid memory issues
train_subset = pref_dataset["train"].select(range(1000))  # Just 1000 examples
test_subset = pref_dataset["test"].select(range(100))     # Just 100 examples

print(f"Train subset: {len(train_subset)} examples")
print(f"Test subset: {len(test_subset)} examples")

Train subset: 1000 examples
Test subset: 100 examples


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from trl import RewardTrainer
import torch

# Clear cache
torch.cuda.empty_cache()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Important for causal LM

# Tokenize function
def tokenize_reward(examples):
    # Process chosen responses
    chosen_inputs = tokenizer(
        examples["chosen"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )

    # Process rejected responses
    rejected_inputs = tokenizer(
        examples["rejected"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )

    return {
        "input_ids_chosen": chosen_inputs["input_ids"],
        "attention_mask_chosen": chosen_inputs["attention_mask"],
        "input_ids_rejected": rejected_inputs["input_ids"],
        "attention_mask_rejected": rejected_inputs["attention_mask"],
    }

# Tokenize the subset
tokenized_dataset = train_subset.map(tokenize_reward, batched=True, remove_columns=train_subset.column_names)
tokenized_test = test_subset.map(tokenize_reward, batched=True, remove_columns=test_subset.column_names)

print(f"Tokenized dataset: {len(tokenized_dataset)} examples")

# Load reward model
reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    torch_dtype=torch.float16
)

# Fix for Falcon architecture
reward_model.config.pad_token_id = tokenizer.pad_token_id

# Training arguments - SIMPLIFIED
training_args = TrainingArguments(
    output_dir="./reward_model",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=True,
    remove_unused_columns=False,
    report_to="none"
)

# Initialize RewardTrainer
reward_trainer = RewardTrainer(
    model=reward_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer
)

# Train (this will take a while, even with subset)
print("Starting reward model training...")
reward_trainer.train()

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenized dataset: 1000 examples


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [ ]:
reward_model.save_pretrained("./reward_model_final")
tokenizer.save_pretrained("./reward_model_final")
print("Reward model saved!")

In [ ]:
from trl import DPOTrainer, DPOConfig

# Prepare data for DPO
def prepare_dpo_data(examples):
    return {
        "prompt": examples["chosen"],  # This is simplified - you'd want to extract just the prompt part
        "chosen": examples["chosen"],
        "rejected": examples["rejected"]
    }

dpo_data = train_subset.select(range(100)).map(prepare_dpo_data)

# DPO configuration
dpo_config = DPOConfig(
    output_dir="./dpo_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# Initialize DPO trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Will use a copy of model as reference
    args=dpo_config,
    train_dataset=dpo_data,
    tokenizer=tokenizer,
)

# Train
dpo_trainer.train()

In [ ]:
def test_model(prompt):
    print(f"\n📝 PROMPT: {prompt}")
    print(f"🔸 RESPONSE: {generate_response(prompt)}")
    print("-" * 50)

# Test with various prompts
test_prompts = [
    "Explain reinforcement learning simply",
    "What is artificial intelligence?",
    "Tell me a joke",
    "Write a short poem about coding"
]

for prompt in test_prompts:
    test_model(prompt)